# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and explore the FAIR^2 dataset using the `mlcroissant` library and the Croissant metadata schema.

### Dataset Source
The dataset is described using a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, including their `@id`s, names, and fields, using the Croissant metadata structure.

> **Note:** Entities are referenced by their unique `@id` (identifier) fields.

In [ ]:
# List all record sets with their @id
from pprint import pprint

print("Available record sets:")
record_sets = metadata.record_sets

for rs in record_sets:
    print(f"- Record set: @id='{rs.id}', name: '{getattr(rs, 'name', 'N/A')}'")
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields:")
        for field in rs.fields:
            print(f"      - Field: @id='{field.id}', name: '{getattr(field, 'name', 'N/A')}', dataType: '{getattr(field, 'data_type', 'N/A')}'")
    if hasattr(rs, 'columns') and rs.columns:
        print("    Columns:")
        for col in rs.columns:
            print(f"      - Column: @id='{col.id}', name: '{getattr(col, 'name', 'N/A')}'")
    print()

## 3. Data Extraction
Load one or more record sets into pandas DataFrames for analysis. All record sets and fields are referenced by their `@id` as per the Croissant schema.

In [ ]:
# Gather record set @ids
all_record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}
print("Loading data for the following record sets:")
pprint(all_record_set_ids)

for record_set_id in all_record_set_ids:
    # Extract records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set @id: {record_set_id} (shape: {dataframes[record_set_id].shape})")
    else:
        print(f"No records found for record set @id: {record_set_id}")

# Display the columns available in the first non-empty record set
if dataframes:
    primary_record_set_id = next(iter(dataframes))
    print(f"\nColumns in first record set (@id {primary_record_set_id}):")
    print(dataframes[primary_record_set_id].columns.tolist())
    dataframes[primary_record_set_id].head()
else:
    print("No data loaded from any record set.")

## 4. Exploratory Data Analysis (EDA)
Explore and process the data. Typical steps include filtering, normalization, and grouping.

**Reference all fields and columns by their `@id`. You may need to inspect the output in previous steps for the available field/column ids.**

In [ ]:
import numpy as np

# Pick the first available record set with data for demonstration
if dataframes:
    record_set_id = primary_record_set_id
    df = dataframes[record_set_id].copy()

    print(f"Working with record set: {record_set_id}")
    print("Columns available:")
    print(df.columns.tolist())
    
    # Example: Try to find a likely numeric field by checking dataframe dtypes or column names
    # For demonstration, use the first column with numeric dtype, else the first column
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        numeric_field_id = df.columns[0]  # fallback
    print(f"Selected numeric field for processing: {numeric_field_id}")

    # Filter records: keep rows where value > threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold] if np.issubdtype(df[numeric_field_id].dtype, np.number) else df.copy()

    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization (z-score) for the numeric field
    if np.issubdtype(df[numeric_field_id].dtype, np.number):
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print("Selected field is not numeric; skipping normalization.")

    # Try grouping by another field
    group_field_candidates = [col for col in df.columns if col != numeric_field_id]
    group_field = group_field_candidates[0] if group_field_candidates else None
    if group_field and group_field in filtered_df.columns and np.issubdtype(df[numeric_field_id].dtype, np.number):
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found or selected field is not numeric.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the selected record set.

In [ ]:
import matplotlib.pyplot as plt

if dataframes:
    # Histogram of numeric field
    if np.issubdtype(df[numeric_field_id].dtype, np.number):
        plt.figure(figsize=(8, 4))
        df[numeric_field_id].hist(bins=20)
        plt.title(f"Distribution of '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
    
    # Boxplot grouped by group_field, if available and numeric
    if group_field and group_field in df.columns and np.issubdtype(df[numeric_field_id].dtype, np.number):
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field_id, by=group_field, grid=False)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We loaded the metadata, listed all record sets and their fields using `@id` references, and performed exploratory data analysis with simple filtering, normalization, and visualization. For further analysis, refer to the detailed field descriptions and extend processing as needed.